# 08 — Graph-structured decoding vs unconstrained decoding

Demonstrate that decoding along $U_q$ with $\lambda_\mathrm{ED}$ weighting differs from unconstrained VAE decoding.

The `GraphDecoder` uses `WaveReconstructionBlock` instances that propagate information along the ArrowSpace graph's smooth directions, gated by the dispersion network. The `ClockGatedGraphDecoder` adds the Barontini entropic clock: decoding tempo is modulated by the spectral schedule, so reconstruction effort increases as the denoising process resolves each mode.

In [ ]:
import sys
sys.path.insert(0, "../src")

import torch
import matplotlib.pyplot as plt
from ald_sc.build_prior import build_arrow_prior
from ald_sc.vae import SpectralVAE
from ald_sc.graph_decoder import GraphDecoder, ClockGatedGraphDecoder
from ald_sc.spectral_schedule import SpectralSchedule

torch.manual_seed(3407)

## 1. Build prior and spectral schedule

In [ ]:
F, q = 32, 8
embeddings = torch.randn(64, F)
prior = build_arrow_prior(embeddings, q=q, k=4)
spec_sched = SpectralSchedule(prior, horizon=1.0)

print(f"Prior: F={prior.F}, q={prior.q}")
print(f"ν (eigenvalues): {prior.eigvals_q[:4].tolist()} ...")
print(f"λ_chart: {prior.lambdas_chart[:4].tolist()} ...")

## 2. Compare decoders on the same latent

In [ ]:
z = torch.randn(4, 4, 8, 8)
c_spec = torch.randn(4, 3 * q)

graph_dec = GraphDecoder(
    latent_channels=4, out_channels=3, feature_dim=F,
    base_channels=32, prior=prior,
)
clock_dec = ClockGatedGraphDecoder(
    latent_channels=4, out_channels=3, feature_dim=F,
    base_channels=32, prior=prior, spectral_schedule=spec_sched,
)

with torch.no_grad():
    x_graph = graph_dec(z, c_spec)
    x_clock_early = clock_dec(z, c_spec, diffusion_time=torch.tensor(0.9))
    x_clock_late = clock_dec(z, c_spec, diffusion_time=torch.tensor(0.1))

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i in range(4):
    axes[0, i].imshow(x_graph[i].permute(1, 2, 0).numpy() * 0.5 + 0.5)
    axes[0, i].set_title("GraphDecoder")
    axes[0, i].axis("off")
    axes[1, i].imshow(x_clock_early[i].permute(1, 2, 0).numpy() * 0.5 + 0.5)
    axes[1, i].set_title("ClockGated (t=0.9)")
    axes[1, i].axis("off")
    axes[2, i].imshow(x_clock_late[i].permute(1, 2, 0).numpy() * 0.5 + 0.5)
    axes[2, i].set_title("ClockGated (t=0.1)")
    axes[2, i].axis("off")
plt.suptitle("Graph decoder vs clock-gated decoder at different diffusion times")
plt.tight_layout()
plt.savefig("../results/08_decoder_comparison.png", dpi=150)
plt.show()

## 3. Clock tempo: how gate strength varies with diffusion time

In [ ]:
ts = torch.linspace(0, 1, 50)
tempos = [spec_sched.alpha_bar_k(t).mean().item() for t in ts]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(ts.numpy(), tempos, linewidth=2)
ax.set_xlabel("Diffusion time t (1=noise, 0=clean)")
ax.set_ylabel(r"Gate strength $\bar\alpha_k(t)$")
ax.set_title("Clock-gated decoding tempo")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../results/08_clock_tempo.png", dpi=150)
plt.show()

## 4. Difference between early and late decoding

In [ ]:
diff = (x_clock_early - x_clock_late).abs().mean()
print(f"Mean |early - late|: {diff:.6f}")
print(f"GraphDecoder vs ClockGated(early): {(x_graph - x_clock_early).abs().mean():.6f}")
print(f"GraphDecoder vs ClockGated(late):  {(x_graph - x_clock_late).abs().mean():.6f}")